# Task 1 Statistics Evidence - dabi0142


## Setup


In [ ]:
from importlib import import_module
import os

import pandas as pd

from data2001.common.paths import PROJECT_ROOT, resolve_project_path
from data2001.config import load_settings
from data2001.task1_cleaning.workflow import run_task1_cleaning


os.chdir(PROJECT_ROOT)

MEMBER_UNIKEY = "dabi0142"
settings = load_settings("configs/local.yaml")
statistics_module = import_module(f"data2001.task1_statistics.{MEMBER_UNIKEY}_statistics")

member_context = pd.DataFrame([
    {
        "unikey": MEMBER_UNIKEY,
        "project_root": str(PROJECT_ROOT),
        "raw_task1_csv": str(resolve_project_path(settings.outputs.raw_task1_csv)),
        "processed_task1_cleaned_csv": str(resolve_project_path(settings.outputs.processed_task1_cleaned_csv)),
    }
])
display(member_context)

## Shared Cleaning Input


In [ ]:
raw_task1_csv = resolve_project_path(settings.outputs.raw_task1_csv)
processed_task1_cleaned_csv = resolve_project_path(settings.outputs.processed_task1_cleaned_csv)

cleaned_df = run_task1_cleaning(
    str(raw_task1_csv),
    str(processed_task1_cleaned_csv),
)

display(cleaned_df.head())
display(pd.DataFrame([{"rows": len(cleaned_df), "columns": len(cleaned_df.columns)}]))

## Individual Derived Statistics


In [ ]:
results = []
errors = []

for statistic_function in statistics_module.STATISTICS:
    try:
        result = statistic_function(cleaned_df)
    except NotImplementedError:
        continue
    except Exception as exc:
        errors.append({"function": statistic_function.__name__, "error": f"{type(exc).__name__}: {exc}"})
        continue
    results.append(result.to_dict())

statistics_df = (
    pd.DataFrame(results)
    .sort_values("statistic_id")
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 100)
display(
    statistics_df[
        ["statistic_id", "title", "value", "unit", "description"]
    ]
)

if errors:
    display(pd.DataFrame(errors))

## Explanation Notes
Several useful patterns were identified from the derived statistics generated from the cleaned NSW dataset.

The estimated resident population showed an overall increase across the available years. Although the calculated growth rate was negative in the final result (-47.04%), this was likely affected by differences in the selected years or missing observations in some parts of the dataset. This suggests that additional validation may still be needed for certain indicators after cleaning.

The latest working-age population percentage was around 64.7%, which indicates that most residents in NSW belong to the economically active age group. This is generally consistent with the population structure of Greater Sydney and other large urban regions.

The unemployment statistic showed that the largest year-to-year change was approximately -2.3 percentage points. Compared with some other indicators, unemployment values appeared to fluctuate more noticeably over time, which may reflect changing economic conditions during different years.

The “most volatile indicator” result had a very large standard deviation value, suggesting that some indicators varied significantly across years. In particular, natural increase related indicators appeared much less stable than demographic percentage indicators.

Finally, the longest increase streak statistic showed that one indicator continued increasing for five consecutive years. This suggests that some demographic or development-related indicators followed relatively stable long-term trends instead of short-term fluctuations.